# Notebook 02: Menurunkan Masalah Bisnis Menjadi Tujuan Teknis ML

Di `01_business_objective.ipynb`, kita sudah merumuskan **9 fitur produk** yang lahir langsung dari 5 masalah harian pengguna (mulai dari perubahan pola harian yang tidak disadari sampai kelelahan akibat terlalu banyak notifikasi), lengkap dengan 3 sudut pandang pengguna (**Individu**, **Pendamping**, **Klinisi**) dan gerbang keamanan **Jeda (`ABSTAIN`)**.

Namun, fitur produk masih berupa deskripsi kebutuhan pengguna. Agar tim data science bisa membangun model dan mengukurnya secara objektif, kita perlu menerjemahkan setiap fitur tersebut menjadi **spesifikasi teknis terukur**:
1. Apa tipe tugas pembelajarannya? (klasifikasi biner, regresi kuantil, analisis kesintasan, atau aturan deterministik?)
2. Apa variabel target, jendela waktu, dan ukuran keberhasilannya?
3. Bagaimana batas perangkat keras (**laptop 8 GB RAM, CPU**) membatasi arsitektur yang boleh kita pilih?

Langkah pertama kita adalah membaca `config.json` untuk mengunci parameter dasar komputasi dan jendela waktu proyek agar konsisten di seluruh tahapan berikutnya.

In [1]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

with open(ROOT / "config.json", "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

SEED = int(CONFIG["project"]["seed"])
REPORTS_DIR = ROOT / CONFIG["paths"]["reports"]
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Root proyek          : {ROOT}")
print(f"Horizon Pergeseran   : {CONFIG['modeling_defaults']['shift_horizon_days']} hari ke depan")
print(f"Horizon Waktu        : {CONFIG['modeling_defaults']['time_horizon_weeks']} minggu ke depan")
print(f"Strategi Validasi    : {CONFIG['modeling_defaults']['cv_strategy']}")
print(f"Gerbang Abstain      : {CONFIG['modeling_defaults']['abstain_enabled']}")

Root proyek          : .
Horizon Pergeseran   : 14 hari ke depan
Horizon Waktu        : [2, 4] minggu ke depan
Strategi Validasi    : GroupKFold_or_StratifiedGroupKFold_by_subject
Gerbang Abstain      : True


Parameter dasar dari `config.json` telah terkunci:
- **Seed acak = `42`**: menjamin pembagian data dan pelatihan model dapat diulang dengan hasil sama.
- **Anggaran RAM = `8 GB` (`cpu`)**: membatasi kita agar memprioritaskan model tabular ringan (*Logistic Regression*, *LightGBM*, regresi *spline*, *CoxPH*), bukan model *deep learning* raksasa.
- **Jendela prediksi = `14 hari` (`shift_horizon_days`)** dan rentang kesintasan **`[2, 4]` minggu** dengan validasi silang **per subjek (`StratifiedGroupKFold`)** supaya rekaman dari pasien yang sama tidak bocor antara data latih dan data uji.

### Langkah 1: Memetakan 9 Fitur Produk ke Tugas Teknis ML

Sebelum mengunduh atau membuka satu pun file data mentah, kita harus memetakan masing-masing dari **9 fitur** ke **peran teknisnya** secara jujur:
- Tidak semua fitur membutuhkan model pembelajaran mesin yang rumit. Misalnya, **Tombol Semangat** adalah kurasi pesan suportif berbasis aturan keamanan bahasa, sedangkan **Cermin**, **Radar Pergeseran**, **Kurva Sehat**, **Jurang Insight**, dan **Waktu** membutuhkan pemodelan statistik atau pembelajaran mesin.
- Kita juga memisahkan **3 peran peringatan** dan **gerbang Jeda (`ABSTAIN`)** ke lapisan **kebijakan keputusan (decision policy)**, bukan dicampur ke pelatihan model dasar, agar skor probabilitas mentah tetap netral dan dapat diaudit.

Tabel di bawah ini menyusun peta jalan teknis sementara untuk ke-9 fitur tersebut sebelum diverifikasi ketersediaan datanya.

In [2]:
import pandas as pd

pd.set_option("display.max_colwidth", None)

feature_specs = [
    {
        "fitur": "Cermin",
        "formulasi_ml": "Skor deviasi perilaku mingguan terhadap baseline pribadi historis + penyesuaian konteks (hari kerja/akhir pekan, masa ujian, kalender Ramadan).",
        "sinyal_dan_batas": "Sinyal pasif ponsel (gerak, ritme, mobilitas abstrak, suara sekitar, metadata komunikasi).",
        "baseline_pembanding": "Median atau rata-rata bergerak historis kausal milik individu (tanpa penyesuaian konteks).",
        "metrik_evaluasi": "Stabilitas skor pada minggu stabil vs peningkatan skor menjelang episode; cek drift & kausalitas.",
        "anti_leakage": "Baseline hanya memakai minggu sebelum minggu berjalan (t < t_current); assert kausalitas waktu.",
        "batas_klaim": "Tingkat A / B (pola pribadi, bukan diagnosis).",
    },
    {
        "fitur": "Kurva Sehat",
        "formulasi_ml": "Regresi kuantil referensi populasi (persentil fitur ritme aktivitas-istirahat berdasarkan usia & jenis kelamin).",
        "sinyal_dan_batas": "Khusus wrist-actigraphy (NHANES referensi; OBF-Psychiatric/Psykose/Depresjon validasi). Dilarang diterapkan ke akselerometer ponsel.",
        "baseline_pembanding": "Kuantil empiris per kelompok usia-jenis kelamin vs B-spline / LightGBM quantile regression.",
        "metrik_evaluasi": "Pinball loss (quantile loss), kalibrasi cakupan persentil, pemisahan kelompok klinis vs kontrol (device-shift check).",
        "anti_leakage": "Split berdasarkan subjek; evaluasi device shift antar-dataset tanpa mencampur target klinis ke referensi NHANES.",
        "batas_klaim": "Tingkat B / C untuk wrist-actigraphy saja; bukan vonis sehat/sakit.",
    },
    {
        "fitur": "Jurang Insight",
        "formulasi_ml": "Skor diskordansi = selisih antara self-report (EMA) aktual dan prediksi EMA dari sinyal pasif + interval ketidakpastian.",
        "sinyal_dan_batas": "Sinyal pasif ponsel berpasangan dengan EMA (CrossCheck, StudentLife); BPRS sebagai PROXY validasi.",
        "baseline_pembanding": "Prediksi rata-rata EMA historis kausal subjek.",
        "metrik_evaluasi": "MAE, RMSE, korelasi/peringkat diskordansi terhadap BPRS PROXY, cakupan interval prediksi.",
        "anti_leakage": "Fitur pasif hanya dari jendela hingga waktu EMA; split GroupKFold per subjek.",
        "batas_klaim": "Dilaporkan sebagai diskordansi (selisih EMA vs sinyal pasif), bukan klaim clinical insight.",
    },
    {
        "fitur": "Deteksi Pergeseran",
        "formulasi_ml": "Klasifikasi biner terkalibrasi & deteksi anomali peringatan dini kambuh/pergeseran (horizon 14 hari ke depan) + gerbang ABSTAIN.",
        "sinyal_dan_batas": "Sinyal pasif CrossCheck (utama) atau StudentLife PROXY (fallback resmi Tingkat A jika label CrossCheck terbatas).",
        "baseline_pembanding": "Aturan ambang deviasi sederhana & LogisticRegression terkalibrasi sebelum RandomForest, LightGBM, XGBoost, CatBoost.",
        "metrik_evaluasi": "PR-AUC, ROC-AUC, Brier Score, Expected Calibration Error (ECE), F1/Recall pada laju alarm palsu terkontrol + cakupan ABSTAIN.",
        "anti_leakage": "GroupKFold / StratifiedGroupKFold per subjek; zona penyangga pra/pasca episode; seluruh scaling, tuning Optuna, kalibrasi di dalam training fold.",
        "batas_klaim": "Tingkat B jika label klinis CrossCheck tersedia (out-of-subject); Tingkat A (PROXY) jika mode fallback.",
    },
    {
        "fitur": "Waktu",
        "formulasi_ml": "Model survival sederhana estimasi peluang/arah waktu menuju pergeseran atau kambuh dalam horizon 2-4 minggu ke depan.",
        "sinyal_dan_batas": "Sinyal pasif longitudinal + riwayat waktu berisiko; wajib berlabel PROXY bila target bukan kejadian klinis nyata.",
        "baseline_pembanding": "Kaplan-Meier non-parametrik sederhana vs CoxPH / survival model ringan.",
        "metrik_evaluasi": "Concordance index (C-index) out-of-subject & kalibrasi probabilitas 2-4 minggu.",
        "anti_leakage": "Split per subjek; tidak memakai informasi kejadian masa depan sebagai kovariat.",
        "batas_klaim": "Tingkat A (PROXY) / Tingkat B tergantung ketersediaan label klinis nyata.",
    },
    {
        "fitur": "Pemicu",
        "formulasi_ml": "Identifikasi urutan prediktif kanal sinyal yang paling awal bergeser menjelang episode.",
        "sinyal_dan_batas": "Kanal gerak, ritme harian, mobilitas abstrak, komunikasi & suara sekitar.",
        "baseline_pembanding": "Uji perbandingan terhadap acakan urutan waktu (time-shuffle baseline).",
        "metrik_evaluasi": "Konsistensi peringkat lead-time kanal & selisih lead-time dibanding acakan waktu.",
        "anti_leakage": "Hanya mengukur sinyal sebelum onset episode (tanpa jendela pasca-onset).",
        "batas_klaim": "Urutan prediktif observasional semata; DILARANG klaim sebab-akibat (kausalitas).",
    },
    {
        "fitur": "Inkonsistensi Antar-Sinyal",
        "formulasi_ml": "Fitur ketidakcocokan arah pergeseran antar-kanal sinyal pada individu yang sama (diverifikasi lewat ablasi & pruning).",
        "sinyal_dan_batas": "Perbandingan lintas-kanal (gerak vs ritme vs mobilitas vs komunikasi/suara).",
        "baseline_pembanding": "Model Deteksi Pergeseran tanpa fitur Inkonsistensi Antar-Sinyal (uji ablasi langsung).",
        "metrik_evaluasi": "Delta PR-AUC / Brier / stabilitas lintas fold saat fitur inkonsistensi ditambahkan vs di-prune.",
        "anti_leakage": "Dihitung murni dari deviasi kausal per kanal pada jendela masa lalu.",
        "batas_klaim": "Tingkat A / B; jika fitur ini tidak menambah nilai atau memperburuk metrik, di-prune dan dilaporkan jujur.",
    },
    {
        "fitur": "Tombol Semangat",
        "formulasi_ml": "Pemilih pesan dukungan deterministik berbasis aturan transparan dari pustaka CSV terkurasi (otonomi, kompetensi, keterhubungan).",
        "sinyal_dan_batas": "Status risiko & konteks non-sensitif; TANPA LLM generatif.",
        "baseline_pembanding": "Pemeriksa aturan deterministik + filter daftar kata terlarang + gerbang keamanan Perlu Perhatian / ABSTAIN.",
        "metrik_evaluasi": "100% lolos audit kata terlarang, 100% deterministik, 100% pengalihan ke pesan netral aman saat Perlu Perhatian / ABSTAIN.",
        "anti_leakage": "Tidak berlaku pelatihan statistik; seluruh aturan transparan dan dapat diaudit.",
        "batas_klaim": "Tingkat A (simulasi aturan transparan); DILARANG klaim efektivitas klinis pesan motivasi.",
    },
    {
        "fitur": "Kebijakan Peringatan",
        "formulasi_ml": "Penetapan ambang keputusan berbeda untuk Individu, Pendamping, dan Klinisi + gerbang ABSTAIN di tingkat episode.",
        "sinyal_dan_batas": "Probabilitas terkalibrasi & ketidakpastian dari Deteksi Pergeseran + status kecukupan data.",
        "baseline_pembanding": "Ambang seragam tunggal tanpa ABSTAIN vs kebijakan 3 peran dengan ABSTAIN.",
        "metrik_evaluasi": "Evaluasi episode (penggabungan alarm beruntun + pencocokan 1-lawan-1): Intent-to-Screen Recall, False Alarm Rate, Median Lead Time, Abstain Rate.",
        "anti_leakage": "Pemilihan ambang peran hanya di dalam training fold; dievaluasi pada subjek holdout fold.",
        "batas_klaim": "Tingkat A / B (kebijakan ambang operasional).",
    },
]

features_df = pd.DataFrame(feature_specs)
display(features_df)

,fitur,formulasi_ml,sinyal_dan_batas,baseline_pembanding,metrik_evaluasi,anti_leakage,batas_klaim
0,Cermin,"Skor deviasi perilaku mingguan terhadap baseline pribadi historis + penyesuaian konteks (hari kerja/akhir pekan, masa ujian, kalender Ramadan).","Sinyal pasif ponsel (gerak, ritme, mobilitas abstrak, suara sekitar, metadata komunikasi).",Median atau rata-rata bergerak historis kausal milik individu (tanpa penyesuaian konteks).,Stabilitas skor pada minggu stabil vs peningkatan skor menjelang episode; cek drift & kausalitas.,Baseline hanya memakai minggu sebelum minggu berjalan (t < t_current); assert kausalitas waktu.,"Tingkat A / B (pola pribadi, bukan diagnosis)."
1,Kurva Sehat,Regresi kuantil referensi populasi (persentil fitur ritme aktivitas-istirahat berdasarkan usia & jenis kelamin).,Khusus wrist-actigraphy (NHANES referensi; OBF-Psychiatric/Psykose/Depresjon validasi). Dilarang diterapkan ke akselerometer ponsel.,Kuantil empiris per kelompok usia-jenis kelamin vs B-spline / LightGBM quantile regression.,"Pinball loss (quantile loss), kalibrasi cakupan persentil, pemisahan kelompok klinis vs kontrol (device-shift check).",Split berdasarkan subjek; evaluasi device shift antar-dataset tanpa mencampur target klinis ke referensi NHANES.,Tingkat B / C untuk wrist-actigraphy saja; bukan vonis sehat/sakit.
2,Jurang Insight,Skor diskordansi = selisih antara self-report (EMA) aktual dan prediksi EMA dari sinyal pasif + interval ketidakpastian.,"Sinyal pasif ponsel berpasangan dengan EMA (CrossCheck, StudentLife); BPRS sebagai PROXY validasi.",Prediksi rata-rata EMA historis kausal subjek.,"MAE, RMSE, korelasi/peringkat diskordansi terhadap BPRS PROXY, cakupan interval prediksi.",Fitur pasif hanya dari jendela hingga waktu EMA; split GroupKFold per subjek.,"Dilaporkan sebagai diskordansi (selisih EMA vs sinyal pasif), bukan klaim clinical insight."
3,Deteksi Pergeseran,Klasifikasi biner terkalibrasi & deteksi anomali peringatan dini kambuh/pergeseran (horizon 14 hari ke depan) + gerbang ABSTAIN.,Sinyal pasif CrossCheck (utama) atau StudentLife PROXY (fallback resmi Tingkat A jika label CrossCheck terbatas).,"Aturan ambang deviasi sederhana & LogisticRegression terkalibrasi sebelum RandomForest, LightGBM, XGBoost, CatBoost.","PR-AUC, ROC-AUC, Brier Score, Expected Calibration Error (ECE), F1/Recall pada laju alarm palsu terkontrol + cakupan ABSTAIN.","GroupKFold / StratifiedGroupKFold per subjek; zona penyangga pra/pasca episode; seluruh scaling, tuning Optuna, kalibrasi di dalam training fold.",Tingkat B jika label klinis CrossCheck tersedia (out-of-subject); Tingkat A (PROXY) jika mode fallback.
4,Waktu,Model survival sederhana estimasi peluang/arah waktu menuju pergeseran atau kambuh dalam horizon 2-4 minggu ke depan.,Sinyal pasif longitudinal + riwayat waktu berisiko; wajib berlabel PROXY bila target bukan kejadian klinis nyata.,Kaplan-Meier non-parametrik sederhana vs CoxPH / survival model ringan.,Concordance index (C-index) out-of-subject & kalibrasi probabilitas 2-4 minggu.,Split per subjek; tidak memakai informasi kejadian masa depan sebagai kovariat.,Tingkat A (PROXY) / Tingkat B tergantung ketersediaan label klinis nyata.
5,Pemicu,Identifikasi urutan prediktif kanal sinyal yang paling awal bergeser menjelang episode.,"Kanal gerak, ritme harian, mobilitas abstrak, komunikasi & suara sekitar.",Uji perbandingan terhadap acakan urutan waktu (time-shuffle baseline).,Konsistensi peringkat lead-time kanal & selisih lead-time dibanding acakan waktu.,Hanya mengukur sinyal sebelum onset episode (tanpa jendela pasca-onset).,Urutan prediktif observasional semata; DILARANG klaim sebab-akibat (kausalitas).
6,Inkonsistensi Antar-Sinyal,Fitur ketidakcocokan arah pergeseran antar-kanal sinyal pada individu yang sama (diverifikasi lewat ablasi & pruning).,Perbandingan lintas-kanal (gerak vs ritme vs mobilitas vs komunikasi/suara).,Model Deteksi Pergeseran tanpa fitur Inkonsistensi Antar-Sinyal (uji ablasi langsung).,Delta PR-AUC / B

Peta 9 fitur ke tugas teknis kini jelas:
- **Jalur klasifikasi terawasi (2 minggu ke depan)**: **Radar Pergeseran** dan **Inkonsistensi** bertindak sebagai fitur/model utama pendeteksi gejala awal, dibantu **Cermin** sebagai komponen deviasi pribadi.
- **Jalur referensi tanpa label klinis**: **Kurva Sehat** memakai regresi kuantil untuk membaca posisi aktivitas terhadap populasi sehat.
- **Jalur kesintasan & penjelas**: **Waktu** (analisis *survival*), **Pemicu** (*lead-time* per kanal), dan **Jurang Insight** (ketidakcocokan antara beban objektif dan persepsi).
- **Lapisan kebijakan & keamanan**: **3 Peran Alert**, **Gerbang Jeda (`ABSTAIN`)**, dan **Tombol Semangat** berdiri sebagai aturan keputusan dan pagar komunikasi aman.

### Langkah 2: Merumuskan Hipotesis Kerja dan Draf Spesifikasi Teknis

Agar proyek ini bersifat ilmiah (bukan asal mencoba model hingga angkanya tinggi), kita mengunci **5 hipotesis eksplisit (`H1`-`H5`)** sebelum menyentuh data:
- **H1**: Pergeseran perilaku relatif terhadap *baseline* pribadi (Cermin) membawa sinyal awal sebelum hari bergejala berat.
- **H2**: Mengukur ketidakselarasan antar-kanal perilaku (*inconsistency*, misalnya tidur kacau namun aktivitas tetap dipaksa tinggi) menambah daya prediksi dibanding memakai nilai kanal mentah saja.
- **H3**: Kurva persentil aktivitas berbasis populasi umum (NHANES) mampu memetakan posisi aktivitas harian tanpa perlu label klinis pada tahap referensi.
- **H4**: Gerbang ketidakpastian **`ABSTAIN`** dan ambang biaya peran (**Individu**, **Pendamping**, **Klinisi**) mampu menekan alarm palsu per 30 hari dibanding peringatan tanpa filter.
- **H5**: Uji acak waktu dan pemisahan pasien ketat (**GroupKFold / StratifiedGroupKFold by subject**) mencegah kebocoran identitas pasien antar-fold.

Kita tulis semuanya ke `reports/task_spec.md` dalam status **`DRAF`** yang siap diuji terhadap realitas data di notebook berikutnya.

In [3]:
hypotheses_df = pd.DataFrame([
    {
        "hipotesis_id": "H1 (Model Kompleks vs Baseline: Deteksi Pergeseran)",
        "hipotesis_dapat_gagal": "Model tree terkalibrasi (LightGBM/XGBoost/CatBoost) mampu melampaui LogisticRegression atau aturan ambang sederhana pada PR-AUC & Brier Score out-of-subject.",
        "kondisi_gagal_dan_tindakan": "Jika selisih negatif atau tidak stabil antar-fold, pilih model sederhana yang lebih stabil dan laporkan hasilnya apa adanya.",
    },
    {
        "hipotesis_id": "H2 (Nilai Tambah Inkonsistensi Antar-Sinyal)",
        "hipotesis_dapat_gagal": "Fitur inkonsistensi lintas-kanal menambah sinyal prediktif di atas fitur kanal tunggal pada uji ablasi.",
        "kondisi_gagal_dan_tindakan": "Jika ablasi menunjukkan tidak ada perbaikan atau menurunkan stabilitas kalibrasi, lakukan pruning fitur inkonsistensi dari model utama sambil mempertahankan temuan deskriptifnya.",
    },
    {
        "hipotesis_id": "H3 (Device Shift Actigraphy: Kurva Sehat)",
        "hipotesis_dapat_gagal": "Posisi persentil ritme aktivitas-istirahat pada kurva referensi NHANES membedakan kelompok klinis vs kontrol pada OBF/Psykose/Depresjon.",
        "kondisi_gagal_dan_tindakan": "Jika pergeseran perangkat (device shift) mendominasi perbedaan klinis, laporkan batas transferabilitas antar-perangkat secara eksplisit dan jangan paksakan ke sinyal ponsel.",
    },
    {
        "hipotesis_id": "H4 (Diskordansi EMA vs Sinyal Pasif: Jurang Insight)",
        "hipotesis_dapat_gagal": "Model sinyal pasif mampu memprediksi EMA di atas baseline rata-rata EMA historis individu sehingga selisihnya bermakna.",
        "kondisi_gagal_dan_tindakan": "Jika sinyal pasif gagal mengalahkan baseline EMA historis, catat keterbatasan prediktif secara jujur dan hindari klaim berlebihan pada Jurang Insight.",
    },
    {
        "hipotesis_id": "H5 (Urutan Pergeseran Awal: Pemicu)",
        "hipotesis_dapat_gagal": "Urutan kanal yang paling awal bergeser sebelum episode lebih konsisten daripada acakan waktu (time-shuffle).",
        "kondisi_gagal_dan_tindakan": "Jika urutan tidak berbeda dari acakan waktu, laporkan hasil negatif dan hindari menampilkan klaim pola urutan yang menyesatkan.",
    },
])

display(hypotheses_df)

task_spec_lines = [
    "# Task Specification: Proyek Jeda (DRAF)",
    "",
    "> **Status:** `DRAF`: Akan dibekukan secara resmi pada tahap seleksi objek data (`05_data_selection.ipynb`) setelah audit kualitas data selesai.",
    "",
    "## 1. Spesifikasi 9 Fitur Utama",
    "",
    "| Fitur | Formulasi ML | Baseline Pembanding | Metrik Utama | Batas Klaim |",
    "|---|---|---|---|---|",
]
for row in feature_specs:
    task_spec_lines.append(
        f"| **{row['fitur']}** | {row['formulasi_ml']} | {row['baseline_pembanding']} | {row['metrik_evaluasi']} | {row['batas_klaim']} |"
    )

task_spec_lines.extend([
    "",
    "## 2. Hipotesis yang Dapat Gagal",
    "",
    "| ID | Hipotesis | Aturan Jika Gagal |",
    "|---|---|---|",
])
for _, h in hypotheses_df.iterrows():
    task_spec_lines.append(f"| `{h['hipotesis_id']}` | {h['hipotesis_dapat_gagal']} | {h['kondisi_gagal_dan_tindakan']} |")

task_spec_lines.extend([
    "",
    "## 3. Disiplin Anti-Leakage",
    "- Seluruh evaluasi model prediktif memakai pembagian per subjek (`GroupKFold` / `StratifiedGroupKFold`).",
    "- Seluruh fitur baseline pribadi dan inkonsistensi lintas-kanal bersifat strictly causal (`t < t_current`).",
    "- Pemilihan hyperparameter `Optuna`, kalibrasi probabilitas, dan pemilihan ambang peran hanya dijalankan di dalam *training fold*.",
    "- Data sintetis (`SYNTHETIC_SMOKE_TEST_ONLY`) dilarang keras masuk evaluasi atau `metrics_registry.csv`.",
])

task_spec_path = REPORTS_DIR / "task_spec.md"
task_spec_path.write_text("\n".join(task_spec_lines) + "\n", encoding="utf-8")

print(f"Draf task_spec ditulis: {task_spec_path}")

,hipotesis_id,hipotesis_dapat_gagal,kondisi_gagal_dan_tindakan
0,H1 (Model Kompleks vs Baseline — Deteksi Pergeseran),Model tree terkalibrasi (LightGBM/XGBoost/CatBoost) mampu melampaui LogisticRegression atau aturan ambang sederhana pada PR-AUC & Brier Score out-of-subject.,"Jika selisih negatif atau tidak stabil antar-fold, pilih model sederhana yang lebih stabil dan laporkan hasilnya apa adanya."
1,H2 (Nilai Tambah Inkonsistensi Antar-Sinyal),Fitur inkonsistensi lintas-kanal menambah sinyal prediktif di atas fitur kanal tunggal pada uji ablasi.,"Jika ablasi menunjukkan tidak ada perbaikan atau menurunkan stabilitas kalibrasi, lakukan pruning fitur inkonsistensi dari model utama sambil mempertahankan temuan deskriptifnya."
2,H3 (Device Shift Actigraphy — Kurva Sehat),Posisi persentil ritme aktivitas-istirahat pada kurva referensi NHANES membedakan kelompok klinis vs kontrol pada OBF/Psykose/Depresjon.,"Jika pergeseran perangkat (device shift) mendominasi perbedaan klinis, laporkan batas transferabilitas antar-perangkat secara eksplisit dan jangan paksakan ke sinyal ponsel."
3,H4 (Diskordansi EMA vs Sinyal Pasif — Jurang Insight),Model sinyal pasif mampu memprediksi EMA di atas baseline rata-rata EMA historis individu sehingga selisihnya bermakna.,"Jika sinyal pasif gagal mengalahkan baseline EMA historis, catat keterbatasan prediktif secara jujur dan hindari klaim berlebihan pada Jurang Insight."
4,H5 (Urutan Pergeseran Awal — Pemicu),Urutan kanal yang paling awal bergeser sebelum episode lebih konsisten daripada acakan waktu (time-shuffle).,"Jika urutan tidak berbeda dari acakan waktu, laporkan hasil negatif dan hindari menampilkan klaim pola urutan yang menyesatkan."


Draf task_spec ditulis: ./reports\task_spec.md


Dokumen `reports/task_spec.md` berstatus **DRAF** resmi tercatat dengan 5 hipotesis (`H1`-`H5`) yang siap diuji.

Status **DRAF** sangat penting dalam metode ilmiah: kita baru merumuskan *pertanyaan* dan *syarat sukses*, belum mengetahui apakah kolom sensor di dataset mentah benar-benar lengkap, apakah ada label relaps klinis sejati atau harus memakai label proksi (*proxy*), dan seberapa berat ukuran filenya terhadap batas **8 GB RAM**.

### Langkah Berikutnya: Notebook 03: Penemuan Data Mentah (*Data Discovery*)

Agar **draf spesifikasi teknis** di `reports/task_spec.md` bisa dieksekusi, kita membutukan 4 kategori bukti empiris dari dunia nyata:
1. **Data pasif ponsel harian jangka panjang** lengkap dengan skor gejala/EMA (*Ecological Momentary Assessment*) untuk melatih **Cermin**, **Radar Pergeseran**, **Inkonsistensi**, **Jurang Insight**, **Waktu**, dan **Pemicu**.
2. **Data aktivitas populasi umum berskala besar** untuk membangun referensi persentil **Kurva Sehat**.
3. **Data aktigrafi pasien klinis dan kontrol** sebagai pembanding objektif bagi pola gerakan kelompok psikiatri vs kelompok sehat.
4. **Kepastian ukuran disk, estimasi memori RAM, serta ketersediaan kolom label** agar draf kita tidak menabrak batas 8 GB RAM.

Di **Notebook 03 (`03_data_discovery.ipynb`)**, kita belum membersihkan atau melatih apa pun; kita akan memeriksa isi direktori data mentah secara sistematis untuk melihat dataset apa saja yang tersedia dan apakah ke-4 kebutuhan ini terpenuhi.